# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishamerja1903/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [6]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/krishamerja1903/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

cols = ["impressions_90d", "clicks_90d", "avg_position",
        "days_since_last_update", "trend_pct"]

print("DISTRIBUTION AUDIT")
print(df[cols].describe().round(2))

print("\nSkewness:")
print(df[cols].skew().round(2))

print("\nObservation:")
print("Search and engagement signals show uneven distributions, so extreme values should be interpreted carefully.")

DISTRIBUTION AUDIT
       impressions_90d  clicks_90d  avg_position  days_since_last_update  \
count         30000.00    30000.00      30000.00                30000.00   
mean           5200.37       16.10         16.34                   46.10   
std           16838.02       75.08         15.22                   42.08   
min               1.00        0.00          0.00                    1.00   
25%              81.00        0.00          6.20                   20.00   
50%             731.00        1.00         10.80                   20.00   
75%            3615.25        7.00         22.30                  104.00   
max          517715.00     4178.00        245.00                  373.00   

       trend_pct  
count   26612.00  
mean       -4.79  
std       473.86  
min      -100.00  
25%       -62.60  
50%       -33.50  
75%         0.00  
max     44900.00  

Skewness:
impressions_90d           11.38
clicks_90d                18.35
avg_position               1.98
days_since_last_up

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [7]:
print("SIGNAL TESTS")

# Signal 1: content freshness
fresh = df[df["days_since_last_update"] <= 90]["trend_pct"].mean()
old = df[df["days_since_last_update"] > 90]["trend_pct"].mean()

print(f"\n1. Freshness signal")
print(f"Recently updated pages mean trend: {fresh:.2f}%")
print(f"Older pages mean trend: {old:.2f}%")
print("Verdict:", "CONFIRMED" if fresh > old else "OPPOSITE")

# Signal 2: engagement
median_eng = df["engagement_rate"].median()
high_eng = df[df["engagement_rate"] >= median_eng]["trend_pct"].mean()
low_eng = df[df["engagement_rate"] < median_eng]["trend_pct"].mean()

print(f"\n2. Engagement signal")
print(f"Higher-engagement pages mean trend: {high_eng:.2f}%")
print(f"Lower-engagement pages mean trend: {low_eng:.2f}%")
print("Verdict:", "CONFIRMED" if high_eng > low_eng else "OPPOSITE")

# Signal 3: search position
median_pos = df["avg_position"].median()
better_pos = df[df["avg_position"] <= median_pos]["trend_pct"].mean()
worse_pos = df[df["avg_position"] > median_pos]["trend_pct"].mean()

print(f"\n3. Search-position signal")
print(f"Better-position pages mean trend: {better_pos:.2f}%")
print(f"Worse-position pages mean trend: {worse_pos:.2f}%")
print("Verdict:", "CONFIRMED" if better_pos > worse_pos else "OPPOSITE")

SIGNAL TESTS

1. Freshness signal
Recently updated pages mean trend: 0.71%
Older pages mean trend: -15.55%
Verdict: CONFIRMED

2. Engagement signal
Higher-engagement pages mean trend: -4.79%
Lower-engagement pages mean trend: nan%
Verdict: OPPOSITE

3. Search-position signal
Better-position pages mean trend: -11.50%
Worse-position pages mean trend: 1.10%
Verdict: OPPOSITE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:
# Section 3 - The flag-linked test

declining = df[df["trend_direction"] == "down"]
not_declining = df[df["trend_direction"] != "down"]

decline_age = declining["days_since_last_update"].median()
other_age = not_declining["days_since_last_update"].median()

print("FLAG-LINKED TEST")
print(f"Declining pages: {len(declining):,}")
print(f"Other pages: {len(not_declining):,}")

print(f"\nMedian days since update - declining pages: {decline_age:.1f}")
print(f"Median days since update - other pages: {other_age:.1f}")

if decline_age > other_age:
    print("\nVerdict: CONFIRMED")
    print("Declining pages are older by this measure, which supports the flag's assumption.")
elif decline_age < other_age:
    print("\nVerdict: OPPOSITE")
    print("Declining pages are not older by this measure, so the flag's assumption is not supported.")
else:
    print("\nVerdict: MIXED")
    print("Both groups have the same median age, so this test does not clearly support the assumption.")

print("\nThis is observational evidence and does not show that content age causes decline.")

FLAG-LINKED TEST
Declining pages: 16,262
Other pages: 13,738

Median days since update - declining pages: 20.0
Median days since update - other pages: 20.0

Verdict: MIXED
Both groups have the same median age, so this test does not clearly support the assumption.

This is observational evidence and does not show that content age causes decline.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [9]:
# Section 4 - What this means in practice

print("WHAT THIS MEANS IN PRACTICE")
print(
    "Content teams should use multiple historical signals to prioritize pages for human review rather than relying on a single flag. "
    "Freshness showed directional support, while search visibility, search position, and the flag-linked age test did not consistently support the assumed relationship. "
    "These findings should therefore be used as decision-support evidence, not as causal proof."
)

WHAT THIS MEANS IN PRACTICE
Content teams should use multiple historical signals to prioritize pages for human review rather than relying on a single flag. Freshness showed directional support, while search visibility, search position, and the flag-linked age test did not consistently support the assumed relationship. These findings should therefore be used as decision-support evidence, not as causal proof.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.